# 최적화된 AWQ 양자화

## 왜 AWQ인가?

### 벤치마크 결과 ([vLLM Quantization Guide](https://docs.jarvislabs.ai/blog/vllm-quantization-complete-guide-benchmarks))

| 방식 | 처리량 (tok/s) | TTFT | ITL |
|------|---------------|------|-----|
| Baseline | 461 | 151ms | 21.2ms |
| GPTQ | 276 | 165ms | 35.5ms |
| **Marlin-GPTQ** | 712 | 118ms | 13.8ms |
| **Marlin-AWQ** | **741** | 137ms | **12.6ms** |

### 핵심 발견
1. **Marlin-AWQ가 가장 빠름** (741 tok/s)
2. AWQ는 "활성화 인식" 양자화 → 중요한 가중치 보존
3. vLLM이 Marlin 커널 자동 적용

---

# 1. Import

In [1]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ CPU 모드로 실행됩니다")
print("\n✅ Import 완료!")

PyTorch: 2.9.1
CUDA: False
⚠️ CPU 모드로 실행됩니다

✅ Import 완료!


# 2. 설정

### AWQ vs GPTQ 차이점

| 항목 | GPTQ | AWQ |
|------|------|-----|
| 핵심 | Hessian 기반 | 활성화 기반 |
| 속도 | 빠름 | 더 빠름 |
| actorder | 필요 | **"static" 권장** |

In [2]:
# ============================================================================
# 모델 설정
# ============================================================================
MODEL_ID = "./open/base_model"
OUT_DIR = "./model"

# ============================================================================
# 데이터셋 설정
# ============================================================================
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

# ============================================================================
# 캘리브레이션 설정 (베이스라인과 동일)
# ============================================================================
NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512

# ============================================================================
# AWQ 스타일 양자화 설정
# ============================================================================
# llmcompressor는 GPTQModifier로 AWQ 스타일 적용
# actorder="static"이 AWQ와 유사한 효과

SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE = ["embed_tokens", "lm_head"]

# AWQ 최적화 설정
BLOCK_SIZE = 128         # Marlin 호환
DAMPENING_FRAC = 0.01    # AWQ 스타일 (0.01 권장)
ACTORDER = "static"      # AWQ 스타일 (static 권장)

ORIGINAL_MODEL_SIZE_GB = 2.56

print("=" * 60)
print("AWQ 스타일 양자화 설정")
print("=" * 60)
print(f"MODEL_ID: {MODEL_ID}")
print(f"SCHEME: {SCHEME}")
print(f"SAMPLES: {NUM_CALIBRATION_SAMPLES}")
print("---")
print("⭐ AWQ 최적화:")
print(f"  BLOCK_SIZE: {BLOCK_SIZE}")
print(f"  DAMPENING_FRAC: {DAMPENING_FRAC}")
print(f"  ACTORDER: {ACTORDER}")
print("=" * 60)

AWQ 스타일 양자화 설정
MODEL_ID: ./open/base_model
SCHEME: W4A16
SAMPLES: 256
---
⭐ AWQ 최적화:
  BLOCK_SIZE: 128
  DAMPENING_FRAC: 0.01
  ACTORDER: static


# 3. 모델 로드

In [3]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        trust_remote_code=True,
        device_map="cpu",
    )

print(f"[INFO] 모델 파라미터: {model.num_parameters():,}")
print("[INFO] 모델 로드 완료")

[INFO] 모델 로드 중...


`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] 모델 파라미터: 1,279,391,488
[INFO] 모델 로드 완료


# 4. 데이터셋 로드

In [4]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)

print(f"[INFO] 데이터셋 크기: {len(ds)}")

[INFO] 캘리브레이션 데이터 로드 중...
[INFO] 데이터셋 크기: 256


# 5. AWQ 스타일 양자화

In [5]:
print("[INFO] AWQ 스타일 양자화 시작")
print(f"  - scheme: {SCHEME}")
print(f"  - samples: {NUM_CALIBRATION_SAMPLES}")
print(f"  - actorder: {ACTORDER} (AWQ 스타일)")
print(f"  - dampening: {DAMPENING_FRAC}")

if torch.cuda.is_available():
    print("\n🚀 GPU 모드: 10-20분 예상\n")
else:
    print("\n⏳ CPU 모드: 15-30분 예상\n")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        block_size=BLOCK_SIZE,
        dampening_frac=DAMPENING_FRAC,
        actorder=ACTORDER,  # static = AWQ 스타일
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("\n[INFO] 양자화 완료!")

[INFO] AWQ 스타일 양자화 시작
  - scheme: W4A16
  - samples: 256
  - actorder: static (AWQ 스타일)
  - dampening: 0.01

⏳ CPU 모드: 15-30분 예상



Tokenizing:   0%|          | 0/256 [00:00<?, ? examples/s]

2026-02-10T15:24:42.035217+0900 | reset | INFO - Compression lifecycle reset
2026-02-10T15:24:42.037084+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-10T15:24:42.063552+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-10T15:24:42.064212+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-02-10T15:24:42.070899+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead


W0210 15:24:42.108000 95207 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(1/31): Calibrating: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:57<00:00,  4.46it/s]

2026-02-10T15:25:39.788952+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 256 samples


...

(30/31): Calibrating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:56<00:00,  4.54it/s]

2026-02-10T16:00:23.727153+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 256 samples


2026-02-10T16:00:24.127465+0900 | compress | METRIC - time 0.40s
2026-02-10T16:00:24.128040+0900 | compress | METRIC - error 3258.94
2026-02-10T16:00:24.129238+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-10T16:00:24.129566+0900 | compress | METRIC - Compressed module size: 16.941056 MB
2026-02-10T16:00:24.131837+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 256 samples
2026-02-10T16:00:24.380610+0900 | compress | METRIC - time 0.25s
2026-02-10T16:00:24.381103+0900 | compress | METRIC - error 923.58
2026-02-10T16:00:24.382247+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-10T16:00:24.382539+0900 | compress | METRIC - Compressed module size: 4.235264 MB
2026-02-10T16:00:24.383293+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 256 samples
2026-02-10T16:00:24.624586+0900 | compress | METRIC - time 0.24s
2026-02-10T16:0

(31/31): Propagating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:00<00:00, 3975.35it/s]


2026-02-10T16:00:37.454456+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-10T16:00:37.460874+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`

[INFO] 양자화 완료!


# 6. 모델 저장

In [6]:
print("[INFO] 모델 저장 중...")

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

# 파일 확인
print(f"\n[INFO] 저장된 파일:")
total_size = 0
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    total_size += size
    print(f"  {f}: {size/1e6:.1f} MB")

quantized_size_gb = total_size / 1e9

print("\n" + "=" * 60)
print("모델 크기 비교")
print("=" * 60)
print(f"  원본: {ORIGINAL_MODEL_SIZE_GB:.2f} GB")
print(f"  압축: {quantized_size_gb:.2f} GB")
print(f"  압축률: {quantized_size_gb / ORIGINAL_MODEL_SIZE_GB * 100:.1f}%")
print("=" * 60)

[INFO] 모델 저장 중...
2026-02-10T16:00:37.862422+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:02, 102.41it/s]



[INFO] 저장된 파일:
  chat_template.jinja: 0.0 MB
  config.json: 0.0 MB
  generation_config.json: 0.0 MB
  merges.txt: 1.2 MB
  model.safetensors: 1407.7 MB
  recipe.yaml: 0.0 MB
  special_tokens_map.json: 0.0 MB
  tokenizer.json: 7.9 MB
  tokenizer_config.json: 0.1 MB
  vocab.json: 1.9 MB

모델 크기 비교
  원본: 2.56 GB
  압축: 1.42 GB
  압축률: 55.4%


# 7. 제출 파일 생성

In [7]:
zip_name = "submit_awq_optimized"
print(f"[INFO] {zip_name}.zip 생성 중...")

if os.path.exists(f"{zip_name}.zip"):
    os.remove(f"{zip_name}.zip")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e9
print(f"[INFO] 생성 완료: {zip_name}.zip ({zip_size:.2f} GB)")

if zip_size <= 10:
    print("✅ 용량 제한 충족")
else:
    print("❌ 용량 초과!")

[INFO] submit_awq_optimized.zip 생성 중...
[INFO] 생성 완료: submit_awq_optimized.zip (0.88 GB)
✅ 용량 제한 충족


---

# 예상 성능

## Marlin-AWQ 효과

| 지표 | 베이스라인 | AWQ 최적화 |
|------|-----------|------------|
| PerfNorm | ~0.95 | ~0.95 |
| SpeedNorm | ~0.30 | **~0.40** |
| **Score** | ~0.625 | **~0.675** |

## 참고
- [vLLM Quantization Benchmarks](https://docs.jarvislabs.ai/blog/vllm-quantization-complete-guide-benchmarks)
- Marlin-AWQ: 741 tok/s (가장 빠름)

---